<a href="https://colab.research.google.com/github/SuiseiNakagawa/stylistic_drift_analysis/blob/main/data_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data ingestion from my Google Drive

This notebook contains code that I used to scrape my GDrive for past assignment data to be used in the CS156: Pipeline- First Draft assignment.

The script recursively walks and scrapes my Academics folder, where all of my assignments written with Google Docs are located, until all children documents are reached. It then uses the whitelisted keyword [final] to filter for only completed drafts.  

Thus, before running this notebook, I returned to my past assignments and appended [final] to the filenames of assignments I wished to scrape and ensured that other files do not have the keyword in them.

This initial scraping fetches the file id, file name, and created time of each assignment file. Once this data is collected, the notebook used Google APIs to extract the raw text from the files using the file id. Finally, it saves the scraped metadata and raw text to a csv file at the specified location.



In [1]:
#@title Imports and Google API Installation

import os
from collections import Counter
from google.colab import drive
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io
import pandas as pd

!pip install --quiet google-api-python-client


In [2]:
#@title Authenticate Google API and Mount Drive

drive.mount('/content/drive')
auth.authenticate_user()
service = build('drive', 'v3')

folder_id = "16HVbh97AQHapl3PnFBuVHyfgMtm2mEMo"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#@title Helper function to recursively walk a folder and fetch all nested gdocs
def get_all_docs(folder_id):
    all_docs = []

    results = service.files().list(
        q=f"'{folder_id}' in parents",
        fields="files(id, name, mimeType, createdTime)"
    ).execute()

    for f in results.get('files', []):
        if f['mimeType'] == 'application/vnd.google-apps.folder':
            all_docs.extend(get_all_docs(f['id']))
        elif f['mimeType'] == 'application/vnd.google-apps.document':
            if "[final]" in f['name'].lower():
                all_docs.append({
                    "id": f["id"],
                    "name": f["name"],
                    "created_time": f.get("createdTime")
                })

    return all_docs

In [4]:
#@title Helper function to export gdoc content to raw text
def export_doc_text(file_id):
    request = service.files().export_media(
        fileId=file_id,
        mimeType='text/plain'
    )

    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)

    done = False
    while not done:
        _, done = downloader.next_chunk()

    return fh.getvalue().decode('utf-8')

In [5]:
#@title Fetch gdocs with metadata and export to raw text
# This cell requires around 2 minutes to run
docs = get_all_docs(folder_id)

rows = []

for doc in docs:
    try:
        text = export_doc_text(doc["id"])

        rows.append({
            "file_id": doc["id"],
            "name": doc["name"],
            "created_time": doc["created_time"],
            "text_raw": text
        })

    except Exception as e:
        print(f"Failed: {doc['name']}", e)

In [6]:
#@title Preview an extracted row (output is very long)
rows[0]

{'file_id': '1z4dvX8KX5spMNnUNkWoboNuA6doneGrNYJJWlvB2Wno',
 'name': 'Reflection on Track Options [final]',
 'created_time': '2026-01-30T19:51:16.879Z',
 'text_raw': "\ufeffReflection on Track Options\r\n\r\n\r\n\r\n\r\nMinerva University\r\nCP192: Capstone Seminar\r\nProf. L. Odera\r\nJanuary 31, 2026\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n________________\r\n\r\n\r\nReflection on Track Options\r\nProcess Documentation\r\nTo ideate what kind of track to propose, I answered the provided questions below. \r\nOf the tracks that would fulfill the requirements of your major(s), which are the most attractive options? Why? As a double major in CS (Data Science and Statistics) and SS (Empirical Approaches to the Social Sciences), my capstone options are already narrower than those of single majors. Out of the available tracks, I find the Machine Learning-Based Data Science track the most attractive. Not only does this satisfy both my major requirements, but it also combines machin

In [7]:
#@title Save scraped data to csv under /content
df = pd.DataFrame(rows)

# Define the output CSV file path
output_csv_path = '/content/gdrive_scraped_data.csv'

# Save the DataFrame to a CSV file
df.to_csv(output_csv_path, index=False)

print(f"Data successfully saved to {output_csv_path}")

Data successfully saved to /content/gdrive_scraped_data.csv


In [8]:
#@title Display the first few rows of the DataFrame to verify format
display(df.head())

,file_id,name,created_time,text_raw
0,1z4dvX8KX5spMNnUNkWoboNuA6doneGrNYJJWlvB2Wno,Reflection on Track Options [final],2026-01-30T19:51:16.879Z,﻿Reflection on Track Options\r\n\r\n\r\n\r\n\r...
1,19du57lm4DB4yeZuFAlyTAdgp3u-ug8MV4eLIbQNwZk8,LBA [final],2026-02-07T12:24:25.590Z,﻿LBA: Analyzing the Bangle Market of Charminar...
2,16AVkozInKE_ol_qFQMXxQOy3OMK5kHQjDvgRUQ1cUSk,Ethnographic Assignment [final],2026-02-10T13:05:32.687Z,﻿Ethnographic Assignment\r\n\r\n\r\n\r\n\r\nMi...
3,1yNA1ZfpoNLkbVhCHcuCIGBHFT1eCHsjKcmN4_icO7Yw,Elevation Reflection & Engagement [final],2026-01-07T08:37:17.604Z,﻿Elevation Reflection & Engagement \r\nSuisei ...
4,1gBiAfz7LWm-IePjnEvHuelT03Uocw6l9ywwbvunsxX4,Final assignment [final],2025-12-04T22:18:34.515Z,﻿Tab 1\r\n\r\n\r\n\r\n\r\n\r\n\r\nComparative ...
